# Description

It shows the pathways enriched (from the MultiPLIER models) given an LV name (in Settings below).
These "pathways enriched" are a set of limited pathways used during training of this PLIER model.
More pathways might be enriched if using external and more comprehensive databases such as gProfiler or FUMA as shown below.

# Modules loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import re
from pathlib import Path

import pandas as pd

from entity import Trait
import conf

# Settings

In [3]:
LV_NAME = "LV884"

# Paths

In [4]:
OUTPUT_FIGURES_DIR = Path(conf.RESULTS_DIR, "demo", f"{LV_NAME.lower()}").resolve()
display(OUTPUT_FIGURES_DIR)
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PosixPath('/opt/data/results/demo/lv884')

# Load MultiPLIER summary

In [5]:
multiplier_model_summary = pd.read_pickle(conf.MULTIPLIER["MODEL_SUMMARY_FILE"])

In [6]:
multiplier_model_summary.shape

(2157, 5)

In [7]:
multiplier_model_summary.head()

,pathway,LV index,AUC,p-value,FDR
1,KEGG_LYSINE_DEGRADATION,1,0.388059,0.866078,0.956005
2,REACTOME_MRNA_SPLICING,1,0.733057,0.000048,0.000582
3,MIPS_NOP56P_ASSOCIATED_PRE_RRNA_COMPLEX,1,0.680555,0.001628,0.011366
4,KEGG_DNA_REPLICATION,1,0.549473,0.312155,0.539951
5,PID_MYC_ACTIVPATHWAY,1,0.639303,0.021702,0.083739


# LV pathways

In [8]:
lv_pathways = multiplier_model_summary[
    multiplier_model_summary["LV index"].isin((LV_NAME[2:],))
    & (
        (multiplier_model_summary["FDR"] < 0.05)
                | (multiplier_model_summary["AUC"] >= 0.75)
    )
]

In [9]:
lv_pathways.shape

(0, 5)

In [10]:
lv_pathways = lv_pathways[["pathway", "AUC", "FDR"]].sort_values("FDR")

In [11]:
lv_pathways = lv_pathways.assign(AUC=lv_pathways["AUC"].apply(lambda x: f"{x:.2f}"))

In [12]:
lv_pathways = lv_pathways.assign(FDR=lv_pathways["FDR"].apply(lambda x: f"{x:.2e}"))

In [13]:
lv_pathways = lv_pathways.rename(
    columns={
        "pathway": "Pathway",
    }
)

In [14]:
lv_pathways.head()

,Pathway,AUC,FDR


# Load LV data

In [15]:
from data.recount2 import LVAnalysis

In [16]:
lv_obj = LVAnalysis(LV_NAME)

Here I show the top 20 genes for our LV. You can see gene symbols, the LV weight (in column `LV603`) and the cytoband.

In [17]:
lv_obj.lv_genes.head(20)

,gene_name,LV884,gene_band
0,ZNF658,5.685500,9q21.11
1,FBXW4,5.171299,10q24.32
2,PPARGC1B,2.842164,5q32
3,UPF3A,2.405158,13q34
4,ZNF2,1.609075,2q11.1
5,ANKH,1.574047,5p15.2
6,HECW2,1.420468,2q32.3
7,GRIK1,1.419604,21q21.3
8,CACNA1I,1.409713,22q13.1
9,GPHN,1.325580,14q23.3


# Pathway enrichment using external databases

## gProfiler

In [18]:
print(" ".join(lv_obj.lv_genes.head(70)["gene_name"].tolist()))

ZNF658 FBXW4 PPARGC1B UPF3A ZNF2 ANKH HECW2 GRIK1 CACNA1I GPHN SLC35A1 POLE2 SEPSECS ZNF430 CDK9 ZNF135 MCF2 GRM2 JMY MPP5 HERC4 SLCO3A1 ZNF557 WNT7A ZBTB10 ICA1 LPIN2 ZNF485 NRIP3 NSUN7 KIT NR1D2 EPS15 GATAD2B RAD52 NNT B9D2 PLEKHA1 SORCS3 RGS7 KALRN TRDMT1 PRPF3 NAA25 VLDLR IFNAR1 CES4A GTF3C3 HMGN1 ZNF10 IRAK4 CACNA2D3 NUP107 ZNF483 CDK5RAP2 ABCA5 COLQ IGLL3P NUDT9 JAM2 CUL2 NSL1 RGS6 FLVCR2 FRY FAN1 RASAL2 AP4B1 CRLS1 GPR55


Copy/paste the list of genes above and use gProfiler: https://biit.cs.ut.ee/gprofiler/gost

Results URL: https://biit.cs.ut.ee/gplink/l/aDASb4TAzS9

**Notes**: just two pathways barely associated.

## FUMA

In [19]:
# print top genes in module
print("\n".join(lv_obj.lv_genes.head(70)["gene_name"].tolist()))

ZNF658
FBXW4
PPARGC1B
UPF3A
ZNF2
ANKH
HECW2
GRIK1
CACNA1I
GPHN
SLC35A1
POLE2
SEPSECS
ZNF430
CDK9
ZNF135
MCF2
GRM2
JMY
MPP5
HERC4
SLCO3A1
ZNF557
WNT7A
ZBTB10
ICA1
LPIN2
ZNF485
NRIP3
NSUN7
KIT
NR1D2
EPS15
GATAD2B
RAD52
NNT
B9D2
PLEKHA1
SORCS3
RGS7
KALRN
TRDMT1
PRPF3
NAA25
VLDLR
IFNAR1
CES4A
GTF3C3
HMGN1
ZNF10
IRAK4
CACNA2D3
NUP107
ZNF483
CDK5RAP2
ABCA5
COLQ
IGLL3P
NUDT9
JAM2
CUL2
NSL1
RGS6
FLVCR2
FRY
FAN1
RASAL2
AP4B1
CRLS1
GPR55


In [20]:
# save all genes in model to use as background list of genes
lv_obj.lv_genes["gene_name"].to_csv(OUTPUT_FIGURES_DIR / "all_genes.txt", header=None, index=False)

In [21]:
OUTPUT_FIGURES_DIR / "all_genes.txt"

PosixPath('/opt/data/results/demo/lv884/all_genes.txt')

In [22]:
!head /opt/data/results/demo/lv24/all_genes.txt

KCNK7
KRT1
ACER1
ASPRV1
CDHR1
CTNNBIP1
CAPNS2
ELOVL3
KLC3
DSP


In [23]:
!wc -l /opt/data/results/demo/lv24/all_genes.txt

6750 /opt/data/results/demo/lv24/all_genes.txt


Now go to the FUMA GENE2FUNC module here: https://fuma.ctglab.nl/gene2func

1. Paste the list of top genes above and then upload the `all_genes.txt` file.
2. Use a "Title" and click on "Submit"

**Notes:** significantly expressed in two brain tissues and testis. The brain one makes sense because it's the same cell type.